In [1]:
import tkinter as tk
from tkinter import ttk
from tkinter import messagebox

def start_analysis():

    name = name_entry.get()
    sex = sex_var.get()

    if name == "":
        messagebox.showerror("Error", "Please enter your name")
        return

    print("Name:", name)
    print("Sex:", sex)

    # Later we will call webcam function here
    messagebox.showinfo(
        "Success",
        f"Hello {name}! Starting skin analysis..."
    )

root = tk.Tk()

root.title("Personal Color Advisor")
root.geometry("400x300")

title = tk.Label(
    root,
    text="Personal Color Advisor",
    font=("Arial", 16, "bold")
)
title.pack(pady=20)

# Name
tk.Label(root, text="Name").pack()

name_entry = tk.Entry(root, width=30)
name_entry.pack(pady=5)

# Sex
tk.Label(root, text="Sex").pack()

sex_var = tk.StringVar()
sex_var.set("Female")

sex_dropdown = ttk.Combobox(
    root,
    textvariable=sex_var,
    values=["Female", "Male", "Prefer not to say"]
)

sex_dropdown.pack(pady=5)

# Button
start_button = tk.Button(
    root,
    text="Analyze My Skin Tone",
    command=start_analysis
)

start_button.pack(pady=20)

root.mainloop()

Name: Niyati
Sex: Female


In [3]:
import tkinter as tk
from tkinter import ttk, messagebox
import cv2
import numpy as np

# -------------------------------
# SKIN TONE DETECTION FUNCTION
# -------------------------------
def detect_skin_tone():

    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )

    cap = cv2.VideoCapture(0)

    tone = "Unknown"
    undertone = "Unknown"

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(100, 100)
        )

        frame_lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
        overall_L = np.mean(frame_lab[:, :, 0])

        for (x, y, w, h) in faces:

            roi_x1 = x + int(w * 0.30)
            roi_y1 = y + int(h * 0.30)
            roi_x2 = x + int(w * 0.70)
            roi_y2 = y + int(h * 0.70)

            skin_roi = frame[roi_y1:roi_y2, roi_x1:roi_x2]

            if skin_roi.size > 0:

                lab = cv2.cvtColor(skin_roi, cv2.COLOR_BGR2LAB)

                avg_L = np.mean(lab[:, :, 0])
                avg_A = np.mean(lab[:, :, 1])
                avg_B = np.mean(lab[:, :, 2])

                normalized_L = avg_L - overall_L + 128

                # -----------------------
                # SKIN TONE
                # -----------------------
                if normalized_L > 170:
                    tone = "Fair"
                elif normalized_L > 145:
                    tone = "Light"
                elif normalized_L > 120:
                    tone = "Medium"
                elif normalized_L > 95:
                    tone = "Olive"
                else:
                    tone = "Deep"

                # -----------------------
                # UNDERTONE
                # -----------------------
                if avg_B > avg_A + 5:
                    undertone = "Warm"
                elif avg_A > avg_B + 5:
                    undertone = "Cool"
                else:
                    undertone = "Neutral"

                cap.release()
                cv2.destroyAllWindows()

                return tone, undertone

        cv2.imshow("Detecting Skin Tone... Press Q to quit", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

    return tone, undertone


# -------------------------------
# RECOMMENDATION SYSTEM
# -------------------------------
recommendations = {
    ("Fair", "Warm"): ["Peach", "Coral", "Cream", "Gold"],
    ("Fair", "Cool"): ["Lavender", "Navy", "Rose Pink", "Emerald"],
    ("Light", "Warm"): ["Peach", "Beige", "Warm Brown", "Olive"],
    ("Light", "Cool"): ["Blue", "Pink", "Lavender", "Grey"],
    ("Medium", "Warm"): ["Terracotta", "Olive Green", "Camel", "Gold"],
    ("Medium", "Cool"): ["Burgundy", "Plum", "Royal Blue", "Silver"],
    ("Olive", "Warm"): ["Mustard", "Khaki", "Warm Green", "Brown"],
    ("Olive", "Cool"): ["Teal", "Navy", "Grey", "Wine"],
    ("Deep", "Warm"): ["Burnt Orange", "Chocolate", "Gold", "Rust"],
    ("Deep", "Cool"): ["Purple", "Cobalt Blue", "Black", "Silver"]
}


# -------------------------------
# BUTTON FUNCTION
# -------------------------------
def start_analysis():

    name = name_entry.get()
    sex = sex_var.get()

    if name == "":
        messagebox.showerror("Error", "Please enter your name")
        return

    messagebox.showinfo("Info", "Camera will open. Look at it clearly.")

    tone, undertone = detect_skin_tone()

    colors = recommendations.get(
        (tone, undertone),
        ["Black", "White"]
    )

    color_text = "\n".join(colors)

    result = f"""
Hello {name}!

Detected Skin Tone: {tone}
Detected Undertone: {undertone}

Recommended Colors:
{color_text}

You can always explore any colors you like.
Fashion is about confidence and expression.
"""

    messagebox.showinfo("Your Color Profile", result)


# -------------------------------
# GUI SETUP (TKINTER)
# -------------------------------
root = tk.Tk()
root.title("Personal Color Advisor")
root.geometry("420x350")

title = tk.Label(
    root,
    text="Personal Color Advisor",
    font=("Arial", 16, "bold")
)
title.pack(pady=15)

# Name
tk.Label(root, text="Enter Name").pack()
name_entry = tk.Entry(root, width=30)
name_entry.pack(pady=5)

# Sex
tk.Label(root, text="Select Sex").pack()

sex_var = tk.StringVar()
sex_var.set("Prefer not to say")

sex_dropdown = ttk.Combobox(
    root,
    textvariable=sex_var,
    values=["Female", "Male", "Prefer not to say"]
)
sex_dropdown.pack(pady=5)

# Button
start_btn = tk.Button(
    root,
    text="Analyze My Skin Tone",
    command=start_analysis,
    bg="black",
    fg="white"
)
start_btn.pack(pady=20)

root.mainloop()